# A/B testing in R (draft notes)

## A/B testing workflow

1. **Define the objective**
   - Specify the business goal (e.g., conversion, retention)
   - Choose primary and secondary metrics
   - Determine the minimum detectable effect (MDE)
2. **Design the experiment**
   - Define treatment and control conditions
   - Specify inclusion criteria and sampling strategy
   - Perform power analysis to determine sample size
3. **Randomisation**
   - Randomly sample users from the target population
   - Randomly assign users to control and treatment groups
4. **Run and monitor**
   - Ensure data integrity and experiment validity
   - Check for sample ratio mismatch (SRM)
5. **Statistical inference**
   - Estimate treatment effects
   - Test for statistical significance
   - Report confidence intervals and effect sizes

## 1 Statistical framework

### Confusion matrix

| | predicted positive class | predicted negative class | |
| :-: | :-: | :-: | :-: |
| **actual positive class** | True positive<br>$(1-\beta)$ | False negative<br>$(\beta)$ | Sensitivity<br>$\left(\frac{TP}{TP+FN}\right)$ |
| **actual negative class** | False positive<br>$(\alpha)$ | True negative<br>$(1-\alpha)$ | Specificity<br>$\left(\frac{TN}{TN+FP}\right)$ |
| | Precision<br>$\left(\frac{TP}{TP+FP}\right)$ | Negative predictive value<br>$\left(\frac{TN}{TN+FN}\right)$ | Accuracy<br>$\left(\frac{TP+TN}{TP+TN+FP+FN}\right)$ |

### Family-wise error rate

Family-wise error rate captures the idea that as the number of hypothesis tests increases, so does the probability of observing at least one false positive. FWER is the probability of having at least one false positive across all tests, which grows with the number of tested variables, number of tests or fluctuation in the sample.

$$
FWER = 1 - (1-\alpha)^m
$$

Where 
- $\alpha$ denotes the probability of Type I Error (false positive),
- $m$ is the number of tests. 

## 2 Independent sample t-test

The t-test is used when we are interested in the mean outcome.

### t-test

An independent sample t-test assumes
- a random sample
- normally distributed data
- similar group variances

Assess assumptions
- Levene's test for variances
- Shapiro-Wilk or Jarque-Bera for normality
- histograms and QQ plots for the shape of the data

For a general hypothesis test $H_0: \theta = \theta_0$ under standard regularity conditions $W \overset{d}{\to}\mathcal{N}(0,1)$,

$$
W =\frac{\hat{\theta}-\theta_0}{SE(\hat{\theta})}, \qquad SE(\hat{\theta}) = \sqrt{\widehat{Var}(\hat{\theta})}
$$

#### Summary table

| test | statistic | distribution under null | confidence interval |
| :--: | :-------: | :----------: | :-----------------: |
| one sample t-test | $t=\frac{\bar{x}-\mu_0}{s/\sqrt{n}}$ | $$\begin{align*}t&\sim t_{df}, \\ \quad df&=n-1 \end{align*}$$ | $\bar{x}\pm t_{\alpha/2, df} \times \frac{s}{\sqrt{n}}$ |
| two-sample t-test<br>(equal variances) | $$\begin{align*} t&=\frac{\bar{x}_1 - \bar{x}_2}{s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}}, \\ \quad s_p^2&=\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2} \end{align*}$$ | $$\begin{align*} t&\sim t_{df}, \\ \quad df&=n_1+n_2-2 \end{align*}$$ | $(\bar{x}_1-\bar{x}_2)\pm t_{\alpha/2, df} \times s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}$ |
| two-sample t-test<br>(unequal variances) | $t=\frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}}$ | $$\begin{align*} t&\sim t_{df}, \\ \quad df&=\frac{\left(\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}\right)^2}{\frac{(s_1^2/n_1)^2}{n_1-1}+\frac{(s_2^2/n_2)^2}{n_2-1}} \end{align*}$$ | $(\bar{x}_1-\bar{x}_2)\pm t_{\alpha/2, df} \times \sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}$ |
| one sample z-test | $z=\frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}$ | $z\sim \mathcal{N}(0,1)$ | $\bar{x}\pm z_{\alpha/2} \times \frac{\sigma}{\sqrt{n}}$ |
| two-sample z-test | $z=\frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{\sigma_1^2}{n_1}+\frac{\sigma_2^2}{n_2}}}$ | $z\sim \mathcal{N}(0,1)$ | $(\bar{x}_1-\bar{x}_2)\pm z_{\alpha/2} \times \sqrt{\frac{\sigma_1^2}{n_1}+\frac{\sigma_2^2}{n_2}}$ |
| paired t-test | $$\begin{align*} t&=\frac{\bar{d}-0}{s_d/\sqrt{n}}, \\ \quad d_i&=x_{1i}-x_{2i} \end{align*}$$ | $$\begin{align*}t&\sim t_{df}, \\ \quad df&=n-1\end{align*}$$ | $\bar{d}\pm t_{\alpha/2, df} \times \frac{s_d}{\sqrt{n}}$ |

Recall that for $X\perp Y$, $Var(X-Y)=Var(X)+Var(Y)=Var(X+Y)$.

### Cohen's d

Effect size.

$$d = \frac{\bar{x}_1-\bar{x}_2}{s_{\text{pooled}}}$$

Where
- $s_{\text{pooled}}=\sqrt{\frac{s_1^2+s_2^2}{2}}$ for equal sample sizes,
- $s_{\text{pooled}}=\sqrt{\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}}$ for unequal sample sizes.

One-sample effect size uses an assumed mean, $d=\frac{\bar{x}-\mu}{s}$.

| d | effect size |
| :-: | :-: |
| $\approx .2$ | small |
| $\approx .5$ | medium |
| $\approx .8$ | large | 

### Power

Power analysis allows us to compute the unknown, such as the required sample size.

To derive the formula, start with two [CLT](https://www.probabilitycourse.com/chapter7/7_1_2_central_limit_theorem.php) equations $Z_{\alpha/2}=\frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}$ and -$Z_{\beta}=\frac{\bar{x}-\mu_0+\delta}{\sigma/\sqrt{n}}$. Solve for $\bar{x}$, equate the expressions and solve for n, eliminating $\mu_0$ in the process.

$$\begin{align*}
n &= \frac{2\sigma^2 (Z_{\alpha/2} + Z_\beta)^2}{\delta^2} \\
&= \frac{2(Z_{\alpha/2} + Z_\beta)^2}{\left(\frac{\delta}{\sigma}\right)^2} \\
&= \frac{2(Z_{\alpha/2} + Z_\beta)^2}{d^2}
\end{align*}$$

Where 
- $\sigma$ is the unknown population standard deviation,
- $\alpha$ is the probability of a false positive (typically 0.05),
- $Z_{\alpha/2}$ the corresponding critical value for a two-tailed test (typically 1.96),
- $\beta$ is the probability of a false negative (typically 0.2),
- $Z_\beta$ the corresponding critical value (typically 0.84),
- $\delta$ is the effect size,
- $d$ the effect size in standard deviations (Cohen's d).

### Implementation in R

``` R
# Test normality with Shapiro-Wilk
# H0: X ~ N
stats::shapiro.test(x)

# Test normality with Jarque-Bera
# H0: X ~ N
tseries::jarque.bera.test(x)

# Assess group variances
# H0: var(x1) = var(x2)
car::leveneTest(x1 ~ x2, data = df)

# Two-sample t-test 
# H0: E[Y | group = A] = E[Y | group = B]
stats::t.test(
  metric ~ group,
  data = df,
  alternative = "two.sided",
  conf.level = 0.95,
  var.equal = FALSE
)

# Effect size for the two-sample t-test
# Target: standardized mean difference
# H0 (test): E[Y | A] = E[Y | B]
effectsize::cohens_d(
  metric ~ group,
  data = df,
  pooled_sd = FALSE,
  ci = 0.95
)

# Target: delta = mu_B − mu_A
# Effect size: Cohen's d
# NULL the required variable
# Equal sample sizes
pwr::pwr.t.test(
  n = NULL,
  d = 0.2, # Standardized mean difference
  power = 0.8,
  sig.level = 0.05,
  type = "two.sample",
  alternative = "two.sided"
)

# Unequal sample sizes
pwr::pwr.t2n.test(
  n1 = NULL,
  n2 = 200,
  d = 0.2,
  sig.level = 0.05,
  power = 0.8,
  alternative = "two.sided"
)
```

## 3 Proportions test

Use proportions when the KPI is a binary
- conversion
- signup
- churn
- click (or not)
- purchase (or not)

In large samples, the proportion and the t-test tend towards the same quantity.

### The structure of the data

Table the success counts,

|   | success | failure | group margin |
| :-: | :-: | :-: | :-: | 
| **treatment** | $n_{11}$ | $n_{12}$ | $n_t=n_{11}+n_{12}$ |
| **control** | $n_{21}$ | $n_{22}$ | $n_c=n_{21}+n_{22}$ |
| **outcome margin** | $n_s=n_{11}+n_{21}$ | $n_f=n_{12}+n_{22}$ | $n=n_t+n_c=n_s+n_f$ |

Compute the proportions,

|   | success | failure | group margin |
| :-: | :-: | :-: | :-: | 
| **treatment** | $\hat{p}_1=\frac{n_{11}}{n_t}$ | $1-\hat{p}_1=\frac{n_{12}}{n_t}$ | $\hat{p}_t=1$ |
| **control** | $\hat{p}_2=\frac{n_{21}}{n_c}$ | $1-\hat{p}_2=\frac{n_{22}}{n_c}$ | $\hat{p}_c=1$ |

In terms of probabilities,

|   | success ($Y=1$) | failure ($Y=0$) | marginal $X$ |
| :-: | :-: | :-: | :-: |
| **treatment** ($X=1$) | $p_1=P(Y = 1 \mid X = 1)$ | $(1-p_1)=P(Y = 0 \mid X = 1)$ | $P(X=1)=1$ |
| **control** ($X=0$) | $p_2=P(Y = 1 \mid X = 0)$ | $(1-p_2)=P(Y = 0 \mid X = 0)$ | $P(X=0)=1$ |

### z-test for proportions

Two-sample z-test compares proportions between groups. It assumes
- large sample ($np \geq 5$, $n(1-p) \geq 5$), 
- independent observations.

#### Summary table

| test | statistic | distribution | confidence interval |
| :--: | :-------: | :----------: | :-----------------: |
| two-sample proportion test | $z=\frac{\hat p_e-\hat p_c}{\sqrt{\hat p_{pooled}(1-\hat p_{pooled})(\frac{1}{n_c}+\frac{1}{n_e})}}$ | $z\sim N(0,1)$ | $(\hat p_e-\hat p_c)\pm z_{\alpha/2}\sqrt{\frac{\hat p_e(1-\hat p_e)}{n_e}+\frac{\hat p_c(1-\hat p_c)}{n_c}}$ |

### Cohen's h

Effect size for proportions.

$$h = 2\arcsin{(\sqrt{p_1})} - 2\arcsin{(\sqrt{p_2})}$$

| h | effect size |
| :-: | :-: |
| $\approx .2$ | small |
| $\approx .5$ | medium |
| $\approx .8$ | large |

### Risk difference, risk ratio and odds ratio

Risk difference $\delta$ captures the absolute difference. Risk ratio $\rho$ captures the multiplicative effect on the probability scale. Odds ratio $\theta$ captures the multiplicative effect on the odds scale.

#### Risk difference

**Risk difference** is a measure of the effect size in terms of the difference between the groups. In brief, 
$$
\begin{align*}
\text{risk difference} &= \text{risk in treatment} - \text{risk in control} \\
\delta &= p_A-p_B \\
\Rightarrow \hat{\delta} &= \hat{p}_A-\hat{p}_B
\end{align*}
$$

Assume:

- $H_0: \delta = 0$ (i.e. there is no difference between the groups)
- $H_a: \delta \neq 0$ (i.e. this is a two-tailed test where the difference between the groups can be either larger or smaller than 0)
- $\alpha=0.05 \Rightarrow z_{1-\alpha/2} \approx 1.96$ (i.e. 5% of the probability mass falls into the rejection region and this corresponds to the tails 1.96 standard deviations away either side of the mean)

Begin with computing the test statistic, $$z = \frac{\hat{p}_A-\hat{p}_B-\delta}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_t}+\frac{1}{n_c}\right)}}, \quad
\hat{p} = \frac{n_{11}}{n_t}+\frac{n_{21}}{n_c}$$

**Decision rule**: We reject $H_0$ if $|z| > z_{1-\alpha/2}$. In this case, the result is said to be statistically significant.

Next, construct the confidence interval, $$\delta \in \hat{p}_A-\hat{p}_B \pm Z_{1-\alpha} \text{SE}\left(\hat{\delta}\right), \quad
\text{SE}\left(\hat{\delta}\right) = \sqrt{\frac{\hat{p}_A(1-\hat{p}_A)}{n_t} + \frac{\hat{p}_B(1-\hat{p}_B)}{n_c}}$$

**Decision rule**: We reject $H_0$ if $\delta \notin CI_{\delta}$.

#### Risk ratio

**Risk ratio** measures the effect size as a ratio of the same quantities. In brief,
$$
\begin{align*}
\text{risk ratio} &= \frac{\text{risk in treatment}}{\text{risk in control}} \\
\rho &= \frac{p_A}{p_B} \\
\Rightarrow \hat{\rho} &= \frac{\hat{p}_A}{\hat{p}_B}
\end{align*}
$$

Assume:

- $H_0: \rho = 1$ (i.e. there is no difference between the groups)
- $H_a: \rho \neq 1$ (i.e. this is a two-tailed test where the difference between the groups can be either larger or smaller than 0)
- $\alpha=0.05 \Rightarrow z_{1-\alpha/2} \approx 1.96$

Construct the confidence interval,

$$
\rho \in \exp{\left( \ln{\hat{\rho}}\pm Z_{1-\alpha/2}\sqrt{\widehat{\text{SE}}(\ln{\hat{\rho}})} \right)}, \quad
\widehat{\text{SE}}(\log \hat \rho)
= \sqrt{\frac{1 - \hat p_A}{\hat p_A n_t} + \frac{1 - \hat p_B}{\hat p_B n_c}}=\sqrt{\frac{1}{n_{11}} - \frac{1}{n_{11}+n_{12}} + \frac{1}{n_{21}} - \frac{1}{n_{21}+n_{22}}}
$$

Note: Look up the **Haldane-Anscombe correction** if any of the data counts is zero.

**Decision rule**: We reject $H_0$ if $\rho \notin CI_{\rho}$.
#### Odds ratio

**Odds ratio** measures the effect size as the ratio of the odds. In brief,
$$
\begin{align*}
\text{odds ratio} &= \frac{\text{odds in treatment}}{\text{odds in control}} \\
\theta &= \frac{p_A/(1-p_A)}{p_B/(1-p_B)} \\
\Rightarrow \hat{\theta} &= \frac{\hat{p}_A/(1-\hat{p}_A)}{\hat{p}_B/(1-\hat{p}_B)}
\end{align*}
$$

Assume:

- $H_0: \theta = 1$ (i.e. there is no difference between the groups)
- $H_a: \theta \neq 1$ (i.e. this is a two-tailed test where the difference between the groups can be either larger or smaller than 0)
- $\alpha=0.05 \Rightarrow z_{1-\alpha/2} \approx 1.96$

Construct the confidence interval,

$$
\theta \in \exp{\left( \ln{ \hat \theta \pm Z_{1-\alpha/2}\ \widehat{\mathrm{SE}}(\ln{\hat{\theta}}) } \right)}, \quad
\widehat{\text{SE}}(\ln{\hat{\theta}})
= \sqrt{\frac{1}{n_{11}}+\frac{1}{n_{12}}+\frac{1}{n_{21}}+\frac{1}{n_{22}}}.
$$

Note: Look up the **Haldane-Anscombe correction** if any of the data counts is zero.

**Decision rule**: We reject $H_0$ if $\theta \notin CI_{\theta}$.

### Implementation in R

``` R
# Two-sample proportion test
# H0: p_A = p_B
tab <- with(df, table(group, metric))
#tab <- with(df, table(group, as.integer(metric))) # For logical binaries

stats::prop.test(
  x = c(tab["A", "1"], tab["B", "1"]),
  n = c(sum(tab["A", ]), sum(tab["B", ])),
  alternative = "two.sided",
  conf.level = 0.95,
  correct = FALSE
)

# Effect size for the two-sample proportions test
# Target: difference in proportions/risk ratio/odds ratio
# H0 (test): p_A = p_B
effectsize::difference_in_proportions(
  metric ~ group,
  data = df,
  ci = 0.95
)

effectsize::oddsratio(
  metric ~ group,
  data = df,
  ci = 0.95
)

effectsize::riskratio(
  metric ~ group,
  data = df,
  ci = 0.95
)

# # Target: delta_p = p_B − p_A
# Effect size: Cohen's h
# NULL the required variable
# Equal sample sizes
pwr::pwr.2p.test(
  n = NULL,
  h = ES.h(p1 = 0.10, p2 = 0.12), # Converts to Cohen's h
  power = 0.8,
  sig.level = 0.05,
  alternative = "two.sided"
)

# Unequal sample sizes
pwr::pwr.2p2n.test(
    n1 = NULL,
    n2 = 200,
    h = .8, 
    power = .8, 
    sig.level = .05,
    alternative = "two.sided"
)
```

## 4 Non-parametric test

Use the Mann-Whitney rank-sum test for non-normal data. In a nutshell, we are comparing the shapes of the distributions where the null assumes both groups have the same shape.

Typical use
- the outcome is ordinal
- the metric is very heavy-tailed
- a rank-based comparison (as opposed to focus on mean uplift)

### Mann-Whitney U test

For two groups $g \in \{A,B\}$, we compute $U_A$ and $U_B$,

$$\begin{align*}
U_A &= n_An_B+\frac{n_A(n_A+1)}{2}-T_A \\ 
U_B &= n_An_B+\frac{n_B(n_B+1)}{2}-T_B \\
U&=\text{min}(U_A,U_B)
\end{align*}$$

Where $T_g$ denotes the rank sum for group $g$, $n_g$ the number of cases in group $g$.

We can express the rank-sum formula alternatively,

$$
U=\sum_{i=1}^{n_B}\sum_{j=1}^{n_A}
\left[I_{\{Y_{Bi}>Y_{Aj}\}}+\frac12 I_{\{Y_{Bi}=Y_{Aj}\}}\right]
$$

Expected value $\mu_U$, standard error $\sigma_U$ and the z-score $z$:

$$\begin{align*}
\mu_U&=\frac{n_An_B}{2} \\
\sigma_U&=\sqrt{\frac{n_An_B(n_A+n_B+1)}{12}} \\
z&=\frac{U-\mu_U}{\sigma_U}
\end{align*}$$

### Rank biserial correlation

Effect size for non-parametric tests. $r_{rb}\in[-1,1]$.

$$
r_{rb}=2\left[\Pr(Y_B>Y_A)+\frac12\Pr(Y_B=Y_A)\right]-1
$$

### Implementation in R

``` R
# Mann-Whitney / Wilcoxon rank-sum test
# H0: P(Y_A > Y_B) + 0.5 P(Y_A = Y_B) = 0.5
stats::wilcox.test(
  metric ~ group,
  data = df,
  alternative = "two.sided",
  conf.int = TRUE,
  conf.level = 0.95,
  exact = FALSE
)

# Effect size for the Mann-Whitney / Wilcoxon rank-sum test
# Target: rank-biserial correlation / probability-of-superiority scale
# H0 (test): P(Y_B > Y_A) + 0.5 P(Y_B = Y_A) = 0.5
effectsize::rank_biserial(
  metric ~ group,
  data = df,
  ci = 0.95
)

# Target: θ = P(Y_B > Y_A) + 0.5 P(Y_B = Y_A)
MKpower::sim.ssize.wilcox.test(
  n1 = NULL,
  n2 = 200,
  alpha = 0.05,
  power = 0.8,
  alternative = "two.sided",
  distribution = "normal", # Or specify custom distributions
  location.shift = 0.5 # Effect encoded as shift
)
```

## Critical and p values

``` R
# Two-tailed p-value from t-statistic
p_from_t <- function(t_value, df, tails = "two") {
    if (tails == "two") {
        return(2 * pt(-abs(t_value), df))
    } else if (tails == "lower") {
        return(pt(t_value, df, lower.tail = TRUE))
    } else if (tails == "upper") {
        return(pt(t_value, df, lower.tail = FALSE))
    }
}

# Get t-statistic from p-value
t_from_p <- function(p_value, df, tails = "two") {
    if (tails == "two") {
        return(qt(1 - p_value / 2, df))
    } else if (tails == "lower") {
        return(qt(p_value, df))
    } else if (tails == "upper") {
        return(qt(1 - p_value, df))
    }
}

# Two-tailed p-value from z-statistic
p_from_z <- function(z_value, tails = "two") {
    if (tails == "two") {
        return(2 * pnorm(-abs(z_value)))
    } else if (tails == "lower") {
        return(pnorm(z_value, lower.tail = TRUE))
    } else if (tails == "upper") {
        return(pnorm(z_value, lower.tail = FALSE))
    }
}

# Get z-statistic from p-value
z_from_p <- function(p_value, tails = "two") {
    if (tails == "two") {
        return(qnorm(1 - p_value / 2))
    } else if (tails == "lower") {
        return(qnorm(p_value))
    } else if (tails == "upper") {
        return(qnorm(1 - p_value))
    }
}
```

In [1]:
# Libraries
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Power and sample size

In [2]:
## Compute the required sample size n
pwr::pwr.t.test(d = .8, power = 0.8, sig.level = 0.05, type = "one.sample", alternative = "two.sided")


     One-sample t test power calculation 

              n = 14.30276
              d = 0.8
      sig.level = 0.05
          power = 0.8
    alternative = two.sided
